In [ ]:
#import all libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("mortgage_data.csv")   #

# 2. Select target and PD features

target = "BAD"

pd_features = [
    "JOB",
    "YOJ",
    "DEROG",
    "DELINQ",
    "CLAGE",
    "NINQ",
    "CLNO",
    "DEBTINC"
]

X = df[pd_features].copy()
y = df[target].copy()

# 3. Separate numeric and categorical columns
categorical_features = ["JOB"]
numeric_features = ["YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# 4. Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# 5. Logistic regression model

pd_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("logreg", LogisticRegression(max_iter=2000))
])

pd_model.fit(X_train, y_train)


# 6. Predict PD for all rows

df["PD_model"] = pd_model.predict_proba(X)[:, 1]

print(df["PD_model"].head())

0    0.218903
1    0.479655
2    0.171342
3    0.122876
4    0.101259
Name: PD_model, dtype: float64


In [ ]:
#  fitted logistic regression object
logreg = pd_model.named_steps["logreg"]

#  transformed feature names
feature_names = pd_model.named_steps["preprocessor"].get_feature_names_out()

# Intercept and coefficients
intercept = logreg.intercept_[0]
coefficients = logreg.coef_[0]

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values(by="coefficient", ascending=False)

print("Intercept:", intercept)
print(coef_table)

Intercept: -1.654357991062891
             feature  coefficient
10    cat__JOB_Sales     1.130236
2        num__DELINQ     0.872936
11     cat__JOB_Self     0.623149
6       num__DEBTINC     0.501494
1         num__DEROG     0.482191
4          num__NINQ     0.218880
8     cat__JOB_Other     0.040960
9   cat__JOB_ProfExe     0.034361
0           num__YOJ    -0.050455
5          num__CLNO    -0.230025
3         num__CLAGE    -0.494879
7    cat__JOB_Office    -0.680751


In [ ]:
from scipy.stats import norm


# 1. LGD assumption

df["LGD"] = 0.40  


# 2. EAD

df["EAD"] = df["MORTDUE"].fillna(0)


# 3. Basel asset correlation from section 31.14

R = 0.15

# Avoid exact 0 or 1 PDs
eps = 1e-10
df["PD_model"] = df["PD_model"].clip(eps, 1 - eps)


# 4. Capital requirement K

df["K"] = df["LGD"] * (
    norm.cdf(
        (norm.ppf(df["PD_model"]) + np.sqrt(R) * norm.ppf(0.999)) / np.sqrt(1 - R)
    ) - df["PD_model"]
)


# 5. RWA and capital

df["RWA_IRB"] = 12.5 * df["K"] * df["EAD"]
df["Capital_IRB"] = 0.08 * df["RWA_IRB"]

# check identity
df["Capital_check"] = df["K"] * df["EAD"]

print(df[["PD_model", "LGD", "EAD", "K", "RWA_IRB", "Capital_IRB"]].head())

print("Total IRB RWA:", df["RWA_IRB"].sum())
print("Total IRB Capital:", df["Capital_IRB"].sum())

   PD_model  LGD      EAD         K        RWA_IRB   Capital_IRB
0  0.218903  0.4  25860.0  0.182843   59104.005590   4728.320447
1  0.479655  0.4  70053.0  0.165351  144791.797779  11583.343822
2  0.171342  0.4  13500.0  0.173871   29340.700383   2347.256031
3  0.122876  0.4      0.0  0.157099       0.000000      0.000000
4  0.101259  0.4  97800.0  0.146086  178590.547813  14287.243825
Total IRB RWA: 718416552.9833648
Total IRB Capital: 57473324.23866919


In [ ]:
# 6. Save file with PD values
df.to_csv("Mortgage_with_PD.csv", index=False)